In [1]:
import pandas as pd
import os
import glob

In [2]:
path = r'../data/final/tables/annotations/filled' # use your path
all_files = glob.glob(os.path.join(path, "*.xlsx"))

df_from_each_file = (pd.read_excel(f, skiprows = 3) for f in all_files)
df = pd.concat(df_from_each_file, ignore_index=True)

c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [3]:
df.dropna(subset=['id'], inplace=True)

In [4]:
# Keeping only evaluation of existing rows 

df['id'] = df['id'].astype(str)
df['id_general'] = df['id'].str.replace(r'X_', '', regex=True)

In [5]:
def determine_result_per_group(group):
        has_ja = group['Wert korrekt? (Ja/ Nein)'].eq('Ja').any()
        has_nein = group['Wert korrekt? (Ja/ Nein)'].eq('Nein').any()
        all_na = group['Wert korrekt? (Ja/ Nein)'].isna().all()

        if has_ja and has_nein:
            return '3. Failed extraction: LLM failed to extract all of the metrics correctly'
        elif has_ja:
            return '1. Correct extraction: LLM extracted all metrics correctly'
        elif has_nein:
            return '4. Failed extraction: LLM failed to extract any of the metrics correctly'
        elif all_na:
            return '2. Correct extraction: no extracted metric'
        


def check_correct_result(df, 
                         evaluation_level = 'document' # can be document or row
                         ):
    """
    Groups by 'id' and checks for the following conditions:
    - If at least one 'Ja' and at least one 'Nein' exists -> 'failed to extract all correct metrics'
    - If at least one 'Ja' exists -> 'extracted all correct metrics'
    - If all values are NaN -> 'no extracted metric'
    - Otherwise -> 'incorrect metric'

    Args:
    df (pd.DataFrame): The input DataFrame.

    Returns:
    pd.DataFrame: A DataFrame with 'id' and 'correct_result'.
    """
        
    if evaluation_level == 'document':

        # Group by 'id' and apply the function
        df = df.groupby('id_general').apply(determine_result_per_group, include_groups=False).reset_index(name='correct_result')

    elif evaluation_level == 'row':

        df['correct_result'] = df['Wert korrekt? (Ja/ Nein)'].apply(lambda x: 
                '1. Correct extraction: LLM extracted all metrics correctly' if x == 'Ja' else
                '4. Failed extraction: LLM failed to extract any of the metrics correctly' if x == 'Nein' else
                '2. Correct extraction: no extracted metric' if pd.isna(x) else None
            )

    return df


In [6]:
def evaluate_llm_performance_on_data(df, evaluation_level = 'row'):

    metrics_evaluation = []

    for metric in df['met'].unique():

        keyword_subset = df[df['met'] == metric].copy()  

        keyword_subset['value_match'] = keyword_subset['Wert korrekt? (Ja/ Nein)'].apply(lambda x: 1 if x == 'Ja' or pd.isna(x) else 0)

        evaluation_results = check_correct_result(keyword_subset, evaluation_level = evaluation_level).value_counts('correct_result').reset_index()

        evaluation_results['metric'] = metric

        metrics_evaluation.append(evaluation_results)
    
    return pd.concat(metrics_evaluation, axis=0)
    

In [14]:
def compute_cumulative_percentages(evaluate_data):
    """
    Computes cumulative percentages for each metric based on the correct_result column.

    Args:
    evaluate_data (pd.DataFrame): The input DataFrame.

    Returns:
    pd.DataFrame: A DataFrame with 'metric', 'correct_result', 'count', 'percent', and 'cumulative_percent'.
    """
    # Step 1: Compute total counts per metric
    evaluate_data['metric_total'] = evaluate_data.groupby('metric')['count'].transform('sum')

    # Step 2: Compute percentage for each row
    evaluate_data['percent'] = evaluate_data['count'] / evaluate_data['metric_total'] * 100

    # Step 3: Sort by metric and correct_result (based on its order)
    # Extract the numeric prefix from correct_result for sorting
    evaluate_data['order'] = evaluate_data['correct_result'].str.extract(r'^(\d+)').astype(int)
    evaluate_data = evaluate_data.sort_values(by=['metric', 'order'])

    # Step 4: Compute cumulative sum of percentages within each metric
    evaluate_data['cumulative_percent'] = evaluate_data.groupby('metric')['percent'].cumsum().round(2)

    evaluate_data = evaluate_data.drop(columns=['metric_total', 'order'])

    return evaluate_data

def evaluate_and_format_llm_performance(data, 
                                        evaluation_level, 
                                        cummulative_percentages = True):
    
    if cummulative_percentages: 

        evaluation_results = evaluate_llm_performance_on_data(data, evaluation_level = evaluation_level)
        evaluation_results = compute_cumulative_percentages(evaluation_results)

        evaluation_results = evaluation_results.pivot(index='correct_result', columns='metric', values='cumulative_percent').fillna(0)

        return evaluation_results

    else:
    
        evaluation_results = evaluate_llm_performance_on_data(data, evaluation_level = evaluation_level)

        evaluation_results = evaluation_results.pivot(index='correct_result', columns='metric', values='count').fillna(0)

        return evaluation_results

In [15]:
evaluate_data_per_row = evaluate_and_format_llm_performance(df, evaluation_level = 'row', cummulative_percentages = False)
evaluate_data_per_row_cumulative = evaluate_and_format_llm_performance(df, evaluation_level = 'row', cummulative_percentages = True)

evaluate_data_per_document = evaluate_and_format_llm_performance(df, evaluation_level = 'document', cummulative_percentages = False)
evaluate_data_per_document_cumulative = evaluate_and_format_llm_performance(df, evaluation_level = 'document', cummulative_percentages =True)

In [22]:
evaluate_data_per_row

metric,eg_fok_unit,eg_fok_value,fok_unit,fok_value,gfz_value,gok_unit,gok_value,grundwasser_value,grz_value,hw100_value,hw10_value
correct_result,,,,,,,,,,,
1. Correct extraction: LLM extracted all metrics correctly,9.0,9.0,1.0,1.0,19.0,2.0,2.0,6.0,23.0,5.0,0.0
2. Correct extraction: no extracted metric,516.0,516.0,520.0,520.0,488.0,519.0,519.0,510.0,484.0,526.0,522.0
4. Failed extraction: LLM failed to extract any of the metrics correctly,15.0,15.0,3.0,3.0,36.0,2.0,2.0,6.0,45.0,12.0,1.0


In [23]:
evaluate_data_per_row_cumulative

metric,eg_fok_unit,eg_fok_value,fok_unit,fok_value,gfz_value,gok_unit,gok_value,grundwasser_value,grz_value,hw100_value,hw10_value
correct_result,,,,,,,,,,,
1. Correct extraction: LLM extracted all metrics correctly,1.67,1.67,0.19,0.19,3.50,0.38,0.38,1.15,4.17,0.92,0.00
2. Correct extraction: no extracted metric,97.22,97.22,99.43,99.43,93.37,99.62,99.62,98.85,91.85,97.79,99.81
4. Failed extraction: LLM failed to extract any of the metrics correctly,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00


In [24]:
evaluate_data_per_document_cumulative

metric,eg_fok_unit,eg_fok_value,fok_unit,fok_value,gfz_value,gok_unit,gok_value,grundwasser_value,grz_value,hw100_value,hw10_value
correct_result,,,,,,,,,,,
1. Correct extraction: LLM extracted all metrics correctly,4.92,6.56,0.00,0.00,9.84,3.28,3.28,3.28,16.39,1.64,0.00
2. Correct extraction: no extracted metric,90.16,91.80,96.72,96.72,67.21,98.36,98.36,93.44,67.21,95.08,98.36
3. Failed extraction: LLM failed to extract all of the metrics correctly,91.80,95.08,98.36,98.36,72.13,0.00,0.00,95.08,72.13,96.72,0.00
4. Failed extraction: LLM failed to extract any of the metrics correctly,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00


In [25]:
evaluate_data_per_document

metric,eg_fok_unit,eg_fok_value,fok_unit,fok_value,gfz_value,gok_unit,gok_value,grundwasser_value,grz_value,hw100_value,hw10_value
correct_result,,,,,,,,,,,
1. Correct extraction: LLM extracted all metrics correctly,3.0,4.0,0.0,0.0,6.0,2.0,2.0,2.0,10.0,1.0,0.0
2. Correct extraction: no extracted metric,52.0,52.0,59.0,59.0,35.0,58.0,58.0,55.0,31.0,57.0,60.0
3. Failed extraction: LLM failed to extract all of the metrics correctly,1.0,2.0,1.0,1.0,3.0,0.0,0.0,1.0,3.0,1.0,0.0
4. Failed extraction: LLM failed to extract any of the metrics correctly,5.0,3.0,1.0,1.0,17.0,1.0,1.0,3.0,17.0,2.0,1.0
